# MSIS 522 — HW1: The Complete Data Science Workflow
## Individual Household Electric Power Consumption
**Foster School of Business | University of Washington**  
**Instructor:** Prof. Léonard Boussioux  

---
**Dataset:** UCI Individual Household Electric Power Consumption  
**Task:** Regression — predict `Global_active_power` (kW)  
**Author:** Minseok Choi  


## 0. Setup & Imports

In [ ]:
import os, warnings, json
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
import xgboost as xgb
import shap
import tensorflow as tf
from tensorflow import keras

np.random.seed(42)
tf.random.set_seed(42)
plt.rcParams["figure.dpi"] = 120
sns.set_theme(style="whitegrid")
print("All imports successful.")
print(f"sklearn: {joblib.__version__} | xgboost: {xgb.__version__} | TF: {tf.__version__}")


---
## Part 1: Descriptive Analytics (25 points)
> The goal here is to deeply understand the dataset before modeling — telling a compelling visual story about the data.


### 1.1 Dataset Introduction (5 points)

In [ ]:
# Load the raw dataset
DATA_PATH = "../hw1_data/household_power_consumption.txt"

df_raw = pd.read_csv(DATA_PATH, sep=";", na_values=["?"], low_memory=False)
print(f"Raw shape: {df_raw.shape}")
df_raw.head(3)


In [ ]:
# Parse datetime and cast to numeric
df = df_raw.copy()
df["Datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], format="%d/%m/%Y %H:%M:%S")
df.drop(columns=["Date", "Time"], inplace=True)

num_cols = ["Global_active_power","Global_reactive_power","Voltage",
            "Global_intensity","Sub_metering_1","Sub_metering_2","Sub_metering_3"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Time features
df["hour"]        = df["Datetime"].dt.hour
df["day_of_week"] = df["Datetime"].dt.dayofweek
df["month"]       = df["Datetime"].dt.month
df["is_weekend"]  = (df["day_of_week"] >= 5).astype(int)

missing_before = df_raw.isnull().sum().sum() + (df_raw == "?").sum().sum()
df.dropna(inplace=True)
df.sort_values("Datetime", inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Date range: {df['Datetime'].min().date()} → {df['Datetime'].max().date()}")
print(f"Rows after dropping missing values: {len(df):,} ({missing_before/(len(df_raw)*len(df_raw.columns))*100:.2f}% removed)")
print(f"Features: {df.shape[1]} columns")
print()
print(df.dtypes)


In [ ]:
# Sample 50k rows for tractable training
SAMPLE_N = 50_000
df_sample = df.sample(n=SAMPLE_N, random_state=42).reset_index(drop=True)

FEATURES = ["Global_reactive_power","Voltage","Global_intensity",
            "Sub_metering_1","Sub_metering_2","Sub_metering_3",
            "hour","day_of_week","month","is_weekend"]
TARGET = "Global_active_power"

print(f"Training sample: {SAMPLE_N:,} rows")
print(f"\nFeature summary:")
df_sample[FEATURES + [TARGET]].describe().round(3)


**Dataset Description**

The [UCI Individual Household Electric Power Consumption](https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption) dataset records **minute-by-minute** electrical measurements from a single household in Sceaux, France (December 2006 – November 2010).

| Attribute | Value |
|-----------|-------|
| Source | UCI ML Repository |
| Total observations | ~2.07 million |
| Sampling interval | 1 minute |
| Missing values | ~1.25% (marked `?`) |
| Features used | 10 (6 electrical + 4 temporal) |

**Prediction target:** `Global_active_power` — the household's total minute-averaged active power draw in kilowatts (kW).

**Why this matters:** Accurate household power forecasting enables:
- Grid operators to balance supply and demand in real time  
- Smart meters to support dynamic pricing / demand-response programs  
- Homeowners to identify high-consumption appliances and reduce bills  

This is a **regression** task. All features are numerical (no categorical encoding needed).


### 1.2 Target Distribution (5 points)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_sample[TARGET], bins=80, color="#4C72B0", edgecolor="white", alpha=0.85)
axes[0].set_xlabel("Global Active Power (kW)")
axes[0].set_ylabel("Count")
axes[0].set_title("Histogram — Global Active Power")

df_sample[TARGET].plot.kde(ax=axes[1], color="#C44E52", linewidth=2.5)
axes[1].set_xlabel("Global Active Power (kW)")
axes[1].set_title("KDE — Global Active Power")

plt.suptitle("Distribution of Target Variable: Global Active Power", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print(f"Mean: {df_sample[TARGET].mean():.4f} kW")
print(f"Median: {df_sample[TARGET].median():.4f} kW")
print(f"Std: {df_sample[TARGET].std():.4f} kW")
print(f"Skewness: {df_sample[TARGET].skew():.4f}")


**Interpretation:** The target distribution is strongly **right-skewed** (skewness ≈ 1.5), with the majority of readings below 2 kW corresponding to idle or low-use periods, and a long tail extending to ~10 kW during peak appliance use. The large spike near 0 kW reflects standby/overnight consumption.

Since this is a regression task, class imbalance does not apply. The skewness naturally motivates tree-based models (Random Forest, XGBoost) which split on raw values without distributional assumptions, though the linear baseline also performs well because Global_intensity provides an almost-linear predictor.


### 1.3 Feature Distributions and Relationships (10 points)

In [ ]:
# Plot 1: Average consumption by hour of day
hourly = df_sample.groupby("hour")[TARGET].mean().reset_index()
fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(hourly["hour"], hourly[TARGET], color="#4C72B0", alpha=0.85, edgecolor="white")
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Mean Global Active Power (kW)")
ax.set_title("Plot 1: Average Power Consumption by Hour of Day")
ax.set_xticks(range(0, 24))
ax.axhline(df_sample[TARGET].mean(), color="red", linestyle="--", linewidth=1.2, label="Overall mean")
ax.legend()
plt.tight_layout()
plt.show()


**Interpretation (Plot 1):** Power consumption follows a clear **diurnal pattern** — lowest between 2–6 AM when occupants sleep, rising sharply in the morning (7–9 AM for breakfast/heating), dipping mid-day, then peaking in the evening (18–22 h) when occupants return home and run cooking, entertainment, and lighting loads. This makes `hour` one of the strongest temporal predictors.


In [ ]:
# Plot 2: Monthly boxplot
month_labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
fig, ax = plt.subplots(figsize=(13, 4))
data_by_month = [df_sample[df_sample["month"]==m][TARGET].values for m in range(1,13)]
bp = ax.boxplot(data_by_month, patch_artist=True, notch=True,
                medianprops=dict(color="red", linewidth=2),
                flierprops=dict(marker=".", markersize=2, alpha=0.3))
for patch, color in zip(bp["boxes"], plt.cm.coolwarm(np.linspace(0.1, 0.9, 12))):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_xticklabels(month_labels)
ax.set_xlabel("Month"); ax.set_ylabel("Global Active Power (kW)")
ax.set_title("Plot 2: Power Consumption Distribution by Month")
plt.tight_layout()
plt.show()


**Interpretation (Plot 2):** Winter months (Dec–Feb) show **significantly higher median consumption** and wider interquartile ranges than summer months, consistent with space heating and longer nights driving additional load. Summer months (Jun–Aug) have the lowest medians (~0.8 kW), reflecting mild French weather. This seasonal signal makes `month` a useful feature.


In [ ]:
# Plot 3: Weekday vs Weekend violin
fig, ax = plt.subplots(figsize=(7, 4))
plot_data = df_sample.copy()
plot_data["Day Type"] = plot_data["is_weekend"].map({0:"Weekday", 1:"Weekend"})
sns.violinplot(data=plot_data, x="Day Type", y=TARGET,
               palette=["#4C72B0","#DD8452"], inner="quartile", ax=ax)
ax.set_ylabel("Global Active Power (kW)")
ax.set_title("Plot 3: Power Distribution — Weekday vs. Weekend")
plt.tight_layout()
plt.show()
for label, grp in plot_data.groupby("Day Type")[TARGET]:
    print(f"{label}: median={grp.median():.3f} kW, mean={grp.mean():.3f} kW")


**Interpretation (Plot 3):** Weekends show a **slightly heavier upper tail** and marginally higher median than weekdays. This suggests occupants spend more time at home on weekends, running appliances throughout the day rather than concentrating use in morning and evening commuter windows. The `is_weekend` feature adds a small but consistent signal.


In [ ]:
# Plot 4: Sub-metering stacked bar by hour
sub_hourly = df_sample.groupby("hour")[["Sub_metering_1","Sub_metering_2","Sub_metering_3"]].mean()
fig, ax = plt.subplots(figsize=(13, 4))
sub_hourly.plot(kind="bar", stacked=True, ax=ax,
                color=["#4C72B0","#DD8452","#55A868"], alpha=0.85, edgecolor="white")
ax.set_xlabel("Hour of Day"); ax.set_ylabel("Mean Sub-metering Energy (Wh)")
ax.set_title("Plot 4: Sub-metering Energy Breakdown by Hour")
ax.legend(["SM1 — Kitchen","SM2 — Laundry","SM3 — Water Heater/AC"])
ax.set_xticklabels(range(0,24), rotation=0)
plt.tight_layout()
plt.show()


**Interpretation (Plot 4):** `Sub_metering_3` (electric water heater / AC) dominates across nearly all hours, peaking at ~7 AM (morning shower) and again around 20 h (evening). `Sub_metering_1` (kitchen: dishwasher, oven, microwave) spikes at meal times (8 AM, noon, 7 PM). Laundry (`Sub_metering_2`) shows a relatively flat, low-level profile suggesting habitual scheduling. This breakdown explains why SM3 and the `hour` feature are the top SHAP contributors.


In [ ]:
# Plot 5: Global_intensity scatter vs target (physics check)
fig, ax = plt.subplots(figsize=(7, 4))
sample_sub = df_sample.sample(3000, random_state=42)
sc = ax.scatter(sample_sub["Global_intensity"], sample_sub[TARGET],
                c=sample_sub["Voltage"], cmap="viridis", alpha=0.4, s=8)
plt.colorbar(sc, ax=ax, label="Voltage (V)")
ax.set_xlabel("Global Intensity (A)"); ax.set_ylabel("Global Active Power (kW)")
ax.set_title("Plot 5: Current Intensity vs. Active Power (coloured by Voltage)")
plt.tight_layout()
plt.show()
r = sample_sub[["Global_intensity", TARGET]].corr().iloc[0,1]
print(f"Pearson r(Intensity, ActivePower) = {r:.4f}")


**Interpretation (Plot 5):** The near-perfect **linear relationship** (r ≈ 0.999) between `Global_intensity` and `Global_active_power` is a direct consequence of Ohm's law: P = V × I. The color gradient (Voltage) shows slight deviations from linearity — at the same current, higher voltage yields higher power — explaining why both `Voltage` and `Global_intensity` appear as features. This relationship dominates all models.


### 1.4 Correlation Heatmap (5 points)

In [ ]:
corr_cols = FEATURES + [TARGET]
corr = df_sample[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, ax=ax, vmin=-1, vmax=1,
            annot_kws={"size": 9})
ax.set_title("Pearson Correlation Matrix — Features + Target", fontsize=13)
plt.tight_layout()
plt.show()


**Strongest correlations observed:**

| Pair | r | Implication |
|------|---|-------------|
| `Global_intensity` ↔ `Global_active_power` | ~+0.99 | Physics-based — P ≈ V·I; intensity is the dominant predictor |
| `Global_reactive_power` ↔ `Global_active_power` | ~+0.63 | Reactive power co-varies with active load |
| `Voltage` ↔ `Global_active_power` | ~−0.10 | Weak negative: higher voltage at low-load is a grid regulation effect |
| `Sub_metering_3` ↔ `Global_intensity` | ~+0.35 | Water heater / AC pulls significant current |
| `hour` ↔ `Global_active_power` | ~+0.12 | Temporal pattern adds incremental signal |

The dominant `Global_intensity` correlation means even a simple linear regression will perform well. The remaining features capture variance not explained by intensity alone (reactive loads, time-of-day behavioral patterns, seasonal effects).


---
## Part 2: Predictive Analytics (45 points)

All models use:
- **Train/test split:** 70% / 30%, `random_state=42`  
- **CV:** 5-fold `GridSearchCV`, `scoring="neg_mean_squared_error"`  
- **Metrics:** MAE, RMSE, R²  
- **Pre-trained models loaded** from `./models/` (training takes ~15 min; see `train_models.py` to re-run)


### 2.1 Data Preparation

In [ ]:
# Define features and target
X = df_sample[FEATURES]
y = df_sample[TARGET]

# 70/30 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# StandardScaler for Linear Regression and MLP
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"\nPreprocessing applied:")
print("  • StandardScaler (zero mean, unit variance) → used for Linear Regression and MLP")
print("  • No scaling for tree-based models (invariant to monotone transformations)")
print("  • No encoding needed (all features are numeric)")
print("  • Missing values dropped at load time (~1.25% of raw data)")


### 2.2 Linear Regression Baseline (5 points)

In [ ]:
# Load pre-trained model
lr_model = joblib.load("models/linear_regression.pkl")

# Generate predictions
y_pred_lr = lr_model.predict(X_test_scaled)

# Metrics
mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr   = r2_score(y_test, y_pred_lr)

print("=== Linear Regression (Baseline) ===")
print(f"  MAE  = {mae_lr:.4f} kW")
print(f"  RMSE = {rmse_lr:.4f} kW")
print(f"  R²   = {r2_lr:.4f}")
print()
print("Coefficients (top 5 by absolute magnitude):")
coef_df = pd.DataFrame({"Feature": FEATURES, "Coefficient": lr_model.coef_})
print(coef_df.reindex(coef_df["Coefficient"].abs().sort_values(ascending=False).index).head(5).to_string(index=False))


In [ ]:
# Predicted vs Actual
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test, y_pred_lr, alpha=0.15, s=6, color="#4C72B0")
lims = [y_test.min(), y_test.max()]
ax.plot(lims, lims, "r--", lw=1.5, label="Perfect prediction")
ax.set_xlabel("Actual (kW)"); ax.set_ylabel("Predicted (kW)")
ax.set_title(f"Linear Regression — Predicted vs. Actual\nRMSE={rmse_lr:.4f}, R²={r2_lr:.4f}")
ax.legend()
plt.tight_layout()
plt.show()


**Baseline performance:** Linear Regression achieves an exceptional R² ≈ 0.9986, which is expected given the near-perfect linear correlation between `Global_intensity` and the target (P ≈ V·I). The `Global_intensity` coefficient dominates, as expected from physics. This sets a very high baseline that tree-based models must beat.


### 2.3 Decision Tree / CART (5 points)

In [ ]:
# Training code (shown for reference — loads pre-trained model below)
# dt_grid = {"max_depth": [3, 5, 7, 10], "min_samples_leaf": [5, 10, 20, 50]}
# dt_cv = GridSearchCV(
#     DecisionTreeRegressor(random_state=42),
#     dt_grid, cv=5, scoring="neg_mean_squared_error", n_jobs=-1
# )
# dt_cv.fit(X_train, y_train)

# Load pre-trained best estimator
dt_model = joblib.load("models/decision_tree.pkl")
best_params_all = json.load(open("models/best_params.json"))

y_pred_dt = dt_model.predict(X_test)
mae_dt  = mean_absolute_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))
r2_dt   = r2_score(y_test, y_pred_dt)

print("=== Decision Tree (Best from 5-fold GridSearchCV) ===")
print(f"  Best params: {best_params_all['Decision Tree']}")
print(f"  MAE  = {mae_dt:.4f} kW")
print(f"  RMSE = {rmse_dt:.4f} kW")
print(f"  R²   = {r2_dt:.4f}")


In [ ]:
# Tree visualization — best depth is 10, too deep to display fully.
# We show only the top 3 levels for readability.
fig, ax = plt.subplots(figsize=(16, 5))
plot_tree(dt_model, feature_names=FEATURES, max_depth=3,
          filled=True, rounded=True, fontsize=8, ax=ax,
          impurity=False, precision=2)
ax.set_title("Decision Tree — Top 3 Levels (best max_depth=10, shown truncated)", fontsize=11)
plt.tight_layout()
plt.show()
print("Note: The full tree has max_depth=10, which is too deep to display completely.")
print("The top 3 levels shown reveal that Global_intensity is the root split,")
print("followed by Sub_metering_3, confirming SHAP findings.")


**Results:** Best hyperparameters: `max_depth=10`, `min_samples_leaf=5`. The tree achieves R² = 0.9982, slightly below Linear Regression — this may seem counterintuitive, but deep Decision Trees tend to overfit while the dominant linear relationship in this dataset is better captured by regression. The root split is always on `Global_intensity`, confirming its primacy.

**Note on tree visualization:** The best tree has `max_depth=10`, which renders as ~1,024 leaf nodes — too large to display legibly. Only the top 3 levels are shown above, illustrating the split logic.


### 2.4 Random Forest (10 points)

In [ ]:
# Training code (shown for reference)
# rf_grid = {"n_estimators": [50, 100, 200], "max_depth": [3, 5, 8]}
# rf_cv = GridSearchCV(
#     RandomForestRegressor(random_state=42),
#     rf_grid, cv=5, scoring="neg_mean_squared_error", n_jobs=-1
# )
# rf_cv.fit(X_train, y_train)

# Load pre-trained
rf_model = joblib.load("models/random_forest.pkl")
y_pred_rf = rf_model.predict(X_test)
mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf   = r2_score(y_test, y_pred_rf)

print("=== Random Forest (Best from 5-fold GridSearchCV) ===")
print(f"  Best params: {best_params_all['Random Forest']}")
print(f"  MAE  = {mae_rf:.4f} kW")
print(f"  RMSE = {rmse_rf:.4f} kW")
print(f"  R²   = {r2_rf:.4f}")


In [ ]:
# Predicted vs Actual + Feature Importance
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, y_pred_rf, alpha=0.15, s=6, color="#55A868")
lims = [y_test.min(), y_test.max()]
axes[0].plot(lims, lims, "r--", lw=1.5)
axes[0].set_xlabel("Actual (kW)"); axes[0].set_ylabel("Predicted (kW)")
axes[0].set_title(f"Random Forest — Predicted vs. Actual\nRMSE={rmse_rf:.4f}, R²={r2_rf:.4f}")

importances = rf_model.feature_importances_
imp_df = pd.DataFrame({"Feature": FEATURES, "Importance": importances}).sort_values("Importance")
axes[1].barh(imp_df["Feature"], imp_df["Importance"], color="#55A868", alpha=0.85, edgecolor="white")
axes[1].set_xlabel("Feature Importance (Gini)")
axes[1].set_title("Random Forest Feature Importances")

plt.tight_layout()
plt.show()


**Results:** Best: `n_estimators=200`, `max_depth=8`. RMSE = 0.0423, R² = 0.9984. The ensemble of 200 trees outperforms a single Decision Tree (RMSE 0.0448) by reducing variance. Feature importance confirms `Global_intensity` dominates, with `Sub_metering_3` and `Global_reactive_power` in second and third place.


### 2.5 Boosted Trees — XGBoost (10 points)

In [ ]:
# Training code (shown for reference)
# xgb_grid = {
#     "n_estimators": [50, 100, 200],
#     "max_depth": [3, 4, 5, 6],
#     "learning_rate": [0.01, 0.05, 0.1],
# }
# xgb_cv = GridSearchCV(
#     xgb.XGBRegressor(random_state=42, verbosity=0, tree_method="hist"),
#     xgb_grid, cv=5, scoring="neg_mean_squared_error", n_jobs=-1
# )
# xgb_cv.fit(X_train, y_train)

# Load pre-trained
xgb_model = joblib.load("models/xgboost.pkl")
y_pred_xgb = xgb_model.predict(X_test)
mae_xgb  = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb   = r2_score(y_test, y_pred_xgb)

print("=== XGBoost (Best from 5-fold GridSearchCV) ===")
print(f"  Best params: {best_params_all['XGBoost']}")
print(f"  MAE  = {mae_xgb:.4f} kW")
print(f"  RMSE = {rmse_xgb:.4f} kW")
print(f"  R²   = {r2_xgb:.4f}")


In [ ]:
# Predicted vs Actual
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test, y_pred_xgb, alpha=0.15, s=6, color="#C44E52")
lims = [y_test.min(), y_test.max()]
ax.plot(lims, lims, "r--", lw=1.5)
ax.set_xlabel("Actual (kW)"); ax.set_ylabel("Predicted (kW)")
ax.set_title(f"XGBoost — Predicted vs. Actual\nRMSE={rmse_xgb:.4f}, R²={r2_xgb:.4f}")
plt.tight_layout()
plt.show()


**Results:** Best: `n_estimators=200`, `max_depth=6`, `learning_rate=0.1`. XGBoost achieves the **best performance of all models** — RMSE = 0.0350, R² = 0.9989. The gradient boosting framework sequentially corrects residuals, enabling it to capture nonlinear interactions that Random Forest misses with shallower trees. Three hyperparameters were tuned: number of trees, tree depth, and learning rate.


### 2.6 Neural Network — MLP (10 points)

In [ ]:
# Architecture (shown; model pre-trained and loaded below):
#
# Input (10 features)
#   ↓
# Dense(128, ReLU)
#   ↓
# Dense(128, ReLU)
#   ↓
# Dense(64, ReLU)
#   ↓
# Dense(1, linear)    ← regression output
#
# Optimizer: Adam | Loss: MSE | Epochs: 30 | Batch: 512

mlp_model = tf.keras.models.load_model("models/mlp_model.h5")
mlp_model.summary()


In [ ]:
# Load training history and plot
with open("models/mlp_history.json") as f:
    hist = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(hist["loss"],     label="Train Loss",   color="#4C72B0", linewidth=2)
axes[0].plot(hist["val_loss"], label="Val Loss",     color="#DD8452", linewidth=2, linestyle="--")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE Loss")
axes[0].set_title("MLP Training & Validation Loss")
axes[0].legend()

axes[1].plot(hist["mae"],     label="Train MAE",    color="#4C72B0", linewidth=2)
axes[1].plot(hist["val_mae"], label="Val MAE",      color="#DD8452", linewidth=2, linestyle="--")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("MAE (kW)")
axes[1].set_title("MLP Training & Validation MAE")
axes[1].legend()

plt.suptitle("MLP Neural Network — Training History (30 epochs)", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Test-set evaluation
y_pred_mlp = mlp_model.predict(X_test_scaled, verbose=0).flatten()
mae_mlp  = mean_absolute_error(y_test, y_pred_mlp)
rmse_mlp = np.sqrt(mean_squared_error(y_test, y_pred_mlp))
r2_mlp   = r2_score(y_test, y_pred_mlp)

print("=== MLP Neural Network ===")
print(f"  Architecture: Input(10) → Dense(128,ReLU) → Dense(128,ReLU) → Dense(64,ReLU) → Dense(1)")
print(f"  Optimizer: Adam | Loss: MSE | Epochs: 30 | Batch size: 512")
print(f"  MAE  = {mae_mlp:.4f} kW")
print(f"  RMSE = {rmse_mlp:.4f} kW")
print(f"  R²   = {r2_mlp:.4f}")


**Results:** The MLP converges rapidly (loss plateaus by epoch ~10) with train and validation curves tracking closely, indicating minimal overfitting. Final performance: RMSE = 0.0375, R² = 0.9988 — slightly behind XGBoost but ahead of the Decision Tree. The MLP requires scaled inputs (applied via `StandardScaler`) and significantly more setup than tree models, yet delivers comparable accuracy.


### 2.7 Model Comparison Summary (5 points)

In [ ]:
# Summary table
results = {
    "Model": ["Linear Regression","Decision Tree","Random Forest","XGBoost","MLP Neural Network"],
    "MAE":  [mae_lr, mae_dt, mae_rf, mae_xgb, mae_mlp],
    "RMSE": [rmse_lr, rmse_dt, rmse_rf, rmse_xgb, rmse_mlp],
    "R²":   [r2_lr,  r2_dt,  r2_rf,  r2_xgb,  r2_mlp],
}
results_df = pd.DataFrame(results)
results_df["MAE"]  = results_df["MAE"].round(4)
results_df["RMSE"] = results_df["RMSE"].round(4)
results_df["R²"]   = results_df["R²"].round(4)
print(results_df.to_string(index=False))


In [ ]:
# Bar chart — RMSE comparison
PALETTE = ["#4C72B0","#DD8452","#55A868","#C44E52","#8172B2"]
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(results_df["Model"], results_df["RMSE"], color=PALETTE, edgecolor="white", alpha=0.85)
ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=9)
ax.set_ylabel("RMSE (kW)")
ax.set_title("Model Comparison — RMSE on Test Set (lower is better)")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()


**Discussion:**

**XGBoost (RMSE=0.0350, R²=0.9989)** is the best-performing model, marginally outperforming the MLP and substantially outperforming single-tree methods. This is not surprising: gradient boosting iteratively corrects residuals with shallow trees, making it highly sample-efficient for tabular data with mixed nonlinear interactions.

**Linear Regression** surprisingly ranks second in R² (0.9986), performing better than Decision Tree and close to Random Forest — a testament to the strong linear relationship between current intensity and power. This is an interesting case where domain physics reduces the modeling challenge.

**Trade-offs:**

| Model | Accuracy | Interpretability | Training Speed | Requires Scaling |
|-------|----------|-----------------|----------------|-----------------|
| Linear Regression | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | Yes |
| Decision Tree | ⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | No |
| Random Forest | ⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ | No |
| XGBoost | ⭐⭐⭐⭐⭐ | ⭐⭐ (mitigated by SHAP) | ⭐⭐⭐ | No |
| MLP Neural Network | ⭐⭐⭐⭐ | ⭐ | ⭐⭐ | Yes |

For deployment in a production energy forecasting system, XGBoost is the recommended choice, with SHAP providing post-hoc interpretability for stakeholders.


---
## Part 3: Explainability (10 points)
### 3.1 SHAP Analysis — XGBoost (Best Model)


In [ ]:
# Load pre-computed SHAP values (computed on 500 test-set samples)
explainer   = joblib.load("models/shap_explainer.pkl")
shap_values = np.load("models/shap_values.npy")     # shape: (500, 10)
shap_sample = joblib.load("models/shap_sample.pkl") # shape: (500, 10)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Base value (E[f(x)]): {explainer.expected_value:.4f} kW")
print(f"Features: {FEATURES}")


In [ ]:
# Plot 1: SHAP Bar Plot — Mean Absolute SHAP Values
mean_shap = np.abs(shap_values).mean(axis=0)
shap_imp_df = pd.DataFrame({"Feature": FEATURES, "Mean |SHAP|": mean_shap}).sort_values("Mean |SHAP|")

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(shap_imp_df["Feature"], shap_imp_df["Mean |SHAP|"],
        color="#4C72B0", alpha=0.85, edgecolor="white")
ax.set_xlabel("Mean |SHAP value| — average impact on model output (kW)")
ax.set_title("SHAP Feature Importance — XGBoost\n(Mean Absolute SHAP Values)")
plt.tight_layout()
plt.show()


In [ ]:
# Plot 2: SHAP Beeswarm / Summary Plot
sorted_idx = np.argsort(mean_shap)[::-1]
feat_names_sorted = [FEATURES[i] for i in sorted_idx]

fig, ax = plt.subplots(figsize=(10, 7))
for row_i, fi in enumerate(sorted_idx):
    sv = shap_values[:, fi]
    fv = shap_sample.iloc[:, fi].values
    fv_norm = (fv - fv.min()) / (fv.ptp() + 1e-9)
    jitter = np.random.default_rng(fi).uniform(-0.35, 0.35, len(sv))
    sc = ax.scatter(sv, np.full_like(sv, row_i) + jitter,
                    c=fv_norm, cmap="coolwarm", alpha=0.45, s=10, vmin=0, vmax=1)

ax.set_yticks(range(len(feat_names_sorted)))
ax.set_yticklabels(feat_names_sorted, fontsize=10)
ax.axvline(0, color="black", linewidth=0.9, linestyle="--")
ax.set_xlabel("SHAP value (impact on model output, kW)", fontsize=11)
ax.set_title("SHAP Beeswarm Plot — XGBoost\n(color: blue=low feature value, red=high feature value)", fontsize=11)
plt.colorbar(sc, ax=ax, label="Normalized feature value")
plt.tight_layout()
plt.show()


**Interpretation — SHAP Summary:**

**Strongest features:**
1. `Global_intensity` — by far the most impactful feature. High values (red, right) strongly push predictions up; low values (blue, left) pull them down. This mirrors the physics P = V·I.
2. `Sub_metering_3` — water heater / AC. High usage pushes predictions up significantly.
3. `Global_reactive_power` — captures inductive loads (motors, compressors) beyond what intensity alone explains.
4. `hour` — high hour values (evening, red) push predictions up; early morning (blue) pulls them down.

**Direction of impact:**
- `Global_intensity`, `Sub_metering_3`, `Global_reactive_power`: *positive* — more load = more power
- `Voltage`: *slightly negative* — grid regulation effect (voltage rises when load is low)
- `is_weekend`, `month`: small positive biases for high-consumption periods

**Business insight for decision-makers:** To reduce household power consumption, the most effective interventions target `Sub_metering_3` (water heater scheduling, more efficient HVAC) and shifting high-intensity appliance use away from peak evening hours. The `hour` SHAP values also suggest that time-of-use pricing (higher cost in 18–22 h window) would be most effective at shifting load.


In [ ]:
# Plot 3: Waterfall plot for a high-consumption example
# Find a high-consumption test sample
y_pred_xgb_arr = xgb_model.predict(X_test)
high_idx = np.argmax(y_pred_xgb_arr[:500])   # index within shap_sample
sv_row  = shap_values[high_idx]
base_val = float(explainer.expected_value)
pred_val = base_val + sv_row.sum()
actual_val = y_test.iloc[high_idx] if high_idx < len(y_test) else None

print(f"Selected sample index: {high_idx}")
print(f"Base value (mean prediction): {base_val:.4f} kW")
print(f"SHAP sum:                     {sv_row.sum():.4f} kW")
print(f"Final prediction:             {pred_val:.4f} kW")
if actual_val: print(f"Actual value:                 {actual_val:.4f} kW")

# Waterfall chart
contrib = sorted(zip(FEATURES, sv_row), key=lambda x: abs(x[1]), reverse=True)[:8]
feats_w = [c[0] for c in contrib][::-1]
vals_w  = [c[1] for c in contrib][::-1]
colors_w = ["#C44E52" if v > 0 else "#4C72B0" for v in vals_w]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(feats_w, vals_w, color=colors_w, edgecolor="white", alpha=0.85)
ax.axvline(0, color="black", linewidth=0.9)
ax.set_xlabel("SHAP value (kW contribution)")
ax.set_title(f"SHAP Waterfall — High-Consumption Sample #{high_idx}\n"
             f"Base: {base_val:.3f} kW → Predicted: {pred_val:.3f} kW")
ax.text(0.98, 0.02, "Red = pushes prediction UP\nBlue = pushes prediction DOWN",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=8, color="gray")
plt.tight_layout()
plt.show()


**Waterfall Interpretation:** For this high-consumption sample, `Global_intensity` is the primary driver (large positive SHAP, pushing the prediction well above baseline). `Sub_metering_3` adds further positive contribution, confirming the water heater or AC was running simultaneously. The `Voltage` term pulls slightly downward (as expected under heavy load). This type of instance-level explanation enables a smart-home system to tell the user *exactly which appliance* is responsible for a spike, facilitating targeted energy reduction.


---
## Summary

This notebook demonstrated the complete data science workflow on the UCI Household Electric Power Consumption dataset:

| Stage | Key Finding |
|-------|------------|
| EDA | Strong diurnal/seasonal patterns; Global_intensity dominates all correlations |
| Baseline | Linear Regression achieves R²=0.9986 — physics provides a strong prior |
| Best model | **XGBoost** (RMSE=0.0350, R²=0.9989) via 5-fold GridSearchCV |
| Explainability | SHAP confirms Global_intensity + Sub_metering_3 as top drivers |
| Deployment | Streamlit app with 4 tabs, interactive prediction, and real-time SHAP waterfall |

All code, saved models, and the deployed app are available in the linked GitHub repository.
